### Import modules

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow import keras
# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


### Load Data

In [ ]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
# data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

data = data.drop(columns=['file'])
data['pattern'] = data['pattern'].fillna('None Type')
data.head()

### Configure

In [ ]:
EVALUATING_ENABLED = False
TEMP_TEST_SPLIT = False

In [ ]:
if TEMP_TEST_SPLIT:
    temp_train_data,temp_test_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data['pattern'])
    data = temp_train_data
    temp_test_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_test_data.csv', index=False)
    temp_train_data.to_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/proba_distributions/temp_test_data/temp_train_data.csv', index=False)

### NN Architecture

In [ ]:
TARGET_COLUMN = "pattern"

ARTIFACT_DIR = Path("../models/pattern_nn_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "pattern_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
ENCODER_PATH = ARTIFACT_DIR / "label_encoder.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

In [ ]:
def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            # Dense(num_classes, activation="softmax"),
            Dense(num_classes, activation=None),
        ]
    )

@keras.utils.register_keras_serializable()
class TemperatureScaling(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.temperature = tf.Variable(
            initial_value=1.0,
            trainable=True,
            dtype=tf.float32,
            constraint=lambda t: tf.clip_by_value(t, 1e-6, 100.0),
        )

    def call(self, logits):
        return logits / self.temperature
    def get_config(self):
        return super().get_config()

def build_calibrated_model(base_model: tf.keras.Model) -> tf.keras.Model:
    base_model.trainable = False

    inputs = tf.keras.Input(shape=base_model.input_shape[1:])
    logits = base_model(inputs)

    scaled_logits = TemperatureScaling()(logits)
    outputs = tf.keras.layers.Softmax()(scaled_logits)

    return tf.keras.Model(inputs, outputs)

import numpy as np

def nn_train(data=data):
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)

    X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
        X,
        y_encoded,
        test_size=0.3,
        random_state=42,
        stratify=y_encoded,
    )
    X_val, X_test, y_val_enc, y_test_enc = train_test_split(
        X_temp,
        y_temp_enc,
        test_size=0.5,
        random_state=42,
        stratify=y_temp_enc,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
    y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
    y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)


    if not EVALUATING_ENABLED:
        # X_train = np.vstack([X_train, X_val])
        # y_train = np.vstack([y_train, y_val])
        X_val = np.vstack([X_val, X_test])
        y_val = np.vstack([y_val, y_test])

    def expected_calibration_error(
        probs: np.ndarray,
        y_true: np.ndarray,
        n_bins: int = 15,
    ) -> float:
        confidences = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)
        accuracies = (predictions == y_true).astype(float)

        bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        N = len(y_true)

        for i in range(n_bins):
            bin_lower = bin_boundaries[i]
            bin_upper = bin_boundaries[i + 1]

            in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
            bin_size = np.sum(in_bin)

            if bin_size > 0:
                bin_accuracy = np.mean(accuracies[in_bin])
                bin_confidence = np.mean(confidences[in_bin])

                ece += (bin_size / N) * abs(bin_accuracy - bin_confidence)

        return ece


    model = build_classifier(X_train.shape[1], num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
    )

    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    calibrated_model = build_calibrated_model(model)

    calibrated_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
    )


    calibrated_model.fit(
        X_val,
        y_val,
        epochs=50,
        batch_size=256,
        verbose=0,
    )


    if not EVALUATING_ENABLED:
        # Persist artifacts for downstream inference pipelines
        calibrated_model.save(MODEL_PATH, include_optimizer=True)
        joblib.dump(scaler, SCALER_PATH)
        joblib.dump(label_encoder, ENCODER_PATH)
        metadata = {
            "target_column": TARGET_COLUMN,
            "numeric_features": numeric_features,
            "num_classes": num_classes,
            "label_classes": label_encoder.classes_.tolist(),
        }
        METADATA_PATH.write_text(json.dumps(metadata, indent=2))
        print(f"Saved model to {MODEL_PATH}")
        print(f"Saved scaler to {SCALER_PATH}")
        print(f"Saved label encoder to {ENCODER_PATH}")
        print(f"Saved metadata to {METADATA_PATH}")
    else:
        print("\nEvaluation enabled; not saving model artifacts.")

        test_loss, test_acc, test_top3 = calibrated_model.evaluate(X_test, y_test, verbose=0)
        print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

        y_pred = calibrated_model.predict(X_test)
        y_pred_labels = y_pred.argmax(axis=1)
        report = classification_report(
            y_test_enc,
            y_pred_labels,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T
        summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
        class_breakdown = (
            report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
        )
        probs = calibrated_model.predict(X_test)
        ece = expected_calibration_error(probs, y_test_enc, n_bins=15)

        print(f"ECE: {ece:.4f}")


        print("\nKey metrics:")
        display(summary)
        print("\nTop classes by support:")
        display(class_breakdown)

        raw_preds = model.predict(X_test).argmax(axis=1)
        cal_preds = calibrated_model.predict(X_test).argmax(axis=1)

        print("Accuracy identical:", np.all(raw_preds == cal_preds))
    
    return calibrated_model,scaler,label_encoder

In [ ]:
nn_train(data)

### Logistic Regression classifier

In [ ]:
def lr_train(data=data):
    # Logistic regression stage intentionally skipped per latest workflow requirements.
    # The end-to-end classifier now relies solely on the neural network above.
    numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_features:
        raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

    X = data[numeric_features].fillna(0.0).values
    y = data[TARGET_COLUMN].astype(str).values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(label_encoder.classes_)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.25, random_state=42,stratify=y_encoded
    )

    from sklearn.linear_model import LogisticRegression

    logreg = LogisticRegression(max_iter=1000,n_jobs=-1)

    if EVALUATING_ENABLED:
        logreg.fit(X_train, y_train)
        y_pred = logreg.predict(X_test)
        report = classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T
        summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
        class_breakdown = (
            report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
        )
        display(summary)
    else:
        logreg.fit(X, y)
        # Save model
        ARTIFACT_DIR = Path("../models/pattern_logreg_classifier").resolve()
        ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        joblib.dump(logreg, ARTIFACT_DIR / "logistic_regression_model.joblib")
        joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")
        joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")
        print(f"Saved logistic regression model and artifacts to {ARTIFACT_DIR}")

    return logreg,scaler,label_encoder


In [ ]:
lr_train(data)

### Ensemble training

In [ ]:
from sklearn.model_selection import StratifiedKFold

raw_data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
raw_data['pattern'].fillna('None', inplace=True)

synthetic_data       = raw_data[raw_data['file'].str.contains('pattern',na=False)]
verified_communities = raw_data[~raw_data['file'].str.contains('pattern',na=False)]
synthetic_data = synthetic_data.drop(columns=['file'])
verified_communities = verified_communities.drop(columns=['file'])
def get_folded_splits(fold_count=5,random_state=42,use_only_verified=True,min_samples_per_class=5):
    folded_data = []

    _sd = synthetic_data.copy()
    _vd = verified_communities.copy()

    _vd_c = _vd['pattern'].value_counts()
    _vp_i = _vd_c[_vd_c>=min_samples_per_class].index.tolist()
    _vd   = _vd[_vd['pattern'].isin(_vp_i)]

    skf = StratifiedKFold(
        n_splits=fold_count,
        shuffle=True,
        random_state=random_state
    )

    folds = list(skf.split(_vd, _vd['pattern']))

    for test_fold in range(fold_count):
        val_fold = (test_fold + 1) % fold_count
        train_folds = [
            i for i in range(fold_count)
            if i not in [test_fold, val_fold]
        ]

        train_idx = np.concatenate([folds[i][1] for i in train_folds])
        val_idx   = folds[val_fold][1]
        test_idx  = folds[test_fold][1]

        vd_train = _vd.iloc[train_idx]
        vd_val   = _vd.iloc[val_idx]
        vd_test  = _vd.iloc[test_idx]

        if use_only_verified:
            train_data = vd_train
        else:
            train_data = pd.concat([vd_train, _sd], ignore_index=True)

        folded_data.append(
            (train_data, vd_val, vd_test)
        )

    return folded_data

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
# kfolds = StratifiedKFold(n_splits=4,random_state=42,shuffle=True)
# X = data.drop(columns=['pattern'])
# y = data['pattern']
# folds_idx = kfolds.split(X,y)
# folds=[]

# proba_dist = []

# for idx in folds_idx:
#     train,test = data.iloc[idx[0]],data.iloc[idx[1]]
#     folds.append((train,test))

folds = get_folded_splits(fold_count=5,use_only_verified=True,min_samples_per_class=12)
proba_dist = []

for fold in folds:
    train_data, test_data, val_data = fold
    nn_model,nn_scaler,nn_le = nn_train(train_data)
    nn_scaled_val = nn_scaler.transform(val_data.drop(columns=['pattern']))
    nn_scaled_test = nn_scaler.transform(test_data.drop(columns=['pattern']))
    nn_scaled_train = nn_scaler.transform(train_data.drop(columns=['pattern']))
    nn_proba_val = pd.DataFrame(nn_model.predict(nn_scaled_val),columns=nn_le.classes_)
    nn_proba_test = pd.DataFrame(nn_model.predict(nn_scaled_test),columns=nn_le.classes_)
    nn_proba_train = pd.DataFrame(nn_model.predict(nn_scaled_train),columns=nn_le.classes_)
    nn_proba_val['pattern'] = val_data['pattern'].values
    nn_proba_test['pattern'] = test_data['pattern'].values
    nn_proba_train['pattern'] = train_data['pattern'].values
    
    lr_model,lr_scaler,lr_le = lr_train(train_data)
    lr_scaled_val = lr_scaler.transform(val_data.drop(columns=['pattern']))
    lr_scaled_test = lr_scaler.transform(test_data.drop(columns=['pattern']))
    lr_scaled_train = lr_scaler.transform(train_data.drop(columns=['pattern']))
    lr_proba_val = pd.DataFrame(lr_model.predict_proba(lr_scaled_val),columns=lr_le.classes_)
    lr_proba_test = pd.DataFrame(lr_model.predict_proba(lr_scaled_test),columns=lr_le.classes_)
    lr_proba_train = pd.DataFrame(lr_model.predict_proba(lr_scaled_train),columns=lr_le.classes_)
    lr_proba_val['pattern'] = val_data['pattern'].values
    lr_proba_test['pattern'] = test_data['pattern'].values
    lr_proba_train['pattern'] = train_data['pattern'].values

    numeric_cols = nn_proba_val.select_dtypes(include=['number']).columns.tolist()
    meta_proba_val = pd.DataFrame()
    meta_proba_test = pd.DataFrame()
    meta_proba_train = pd.DataFrame()
    meta_proba_val = (nn_proba_val[numeric_cols] + lr_proba_val[numeric_cols]*0.7)/2
    meta_proba_test = (nn_proba_test[numeric_cols] + lr_proba_test[numeric_cols]*0.7)/2
    meta_proba_train = (nn_proba_train[numeric_cols] + lr_proba_train[numeric_cols]*0.7)/2
    meta_proba_val['pattern'] = val_data['pattern'].values
    meta_proba_test['pattern'] = test_data['pattern'].values
    meta_proba_train['pattern'] = train_data['pattern'].values
    proba_dist.append({
        'train':train_data,
        'val':val_data,
        'test':test_data,
        'nn_proba_val':nn_proba_val,
        'lr_proba_val':lr_proba_val,
        'meta_proba_val':meta_proba_val,
        'nn_proba_test':nn_proba_test,
        'lr_proba_test':lr_proba_test,
        'meta_proba_test':meta_proba_test,
        'nn_proba_train':nn_proba_train,
        'lr_proba_train':lr_proba_train,
        'meta_proba_train':meta_proba_train,
    })

In [ ]:
dist_test = pd.DataFrame(columns=proba_dist[0]['meta_proba_test'].columns)
dist_val = pd.DataFrame(columns=proba_dist[0]['meta_proba_val'].columns)
for fold_proba in proba_dist:
    dist_test = pd.concat([dist_test,fold_proba['meta_proba_test']],ignore_index=True)
    dist_val = pd.concat([dist_val,fold_proba['meta_proba_val']],ignore_index=True)

In [ ]:
def get_prediction(row, class_threshold=0.5,none_threshold=0.5):
    high_prob = row.max()
    if high_prob >= class_threshold:
        max_class = row.idxmax()
        return max_class
    elif high_prob <= none_threshold:
        return 'None Type'
    return "Other"


def classify(class_threshold,none_threshold,data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction, class_threshold=class_threshold,none_threshold=none_threshold, axis=1)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

In [ ]:
best = None
best_class_wise = pd.DataFrame()
for th_t in range(0,100,10):
    for thn in range(0,100,100):
        th = th_t/100
        thn=thn/100
        precision, recall, f1, report_df = classify(th,thn,dist_val)
        
        best_class_wise[str(th)+':'+str(thn)] = report_df['f1-score']

        if best is None or f1 > best[0]:
            best = (f1, th, precision, recall)
        print(f"Threshold: {th} => Precision: {precision}, Recall: {recall}, F1-Score: {f1}")

print("-"*10)
print(f"Best Threshold: {best[1]} => Precision: {best[2]}, Recall: {best[3]}, F1-Score: {best[0]}")
best_class_wise['best_threshold'] = best_class_wise.idxmax(axis=1)

In [ ]:
best_class_wise

In [ ]:
best_df = pd.DataFrame()
best_df['Class Threshold'] = best_class_wise['best_threshold'].apply(lambda x: x.split(":")[0])
best_df['None Threshold'] = best_class_wise['best_threshold'].apply(lambda x: x.split(":")[1])
best_df

In [ ]:
def get_prediction_trained(row):
    high_prob = row.max()
    max_class = row.idxmax()
    if float(best_df['Class Threshold'][max_class]) <= float(high_prob):
        return max_class
    return "Other"


def classify_trained(data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction_trained, axis=1)
    print(f"pricision:{precision} recall:{recall} f1:{f1}")
    display(y_pred.value_counts())

    ignore_class = "Other"
    labels = [c for c in y_true.unique() if c != ignore_class]
    labels.extend([c for c in y_pred.unique() if c != ignore_class])
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

precision, recall, f1, report_df = classify_trained(dist_val)
print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")
with_threshold_report = report_df.T
report_df

In [ ]:
def get_prediction_no_threshold(row):
    max_class = row.idxmax()
    return max_class


def classify_trained_no_threshold(data):
    y_true = data['pattern'].fillna('None Type')
    y_pred = data.drop('pattern', axis=1).apply(get_prediction_no_threshold, axis=1)
    print(y_true)
    print(y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = report['macro avg']
    report_df = pd.DataFrame(report).T
    return macro['precision'], macro['recall'], macro['f1-score'],report_df

precision, recall, f1, report_df = classify_trained_no_threshold(dist_val)
print(f"Precision: {precision}, Recall: {recall}, F1-Score: {f1}")
without_threshold_report = report_df.T
report_df

In [ ]:
with_threshold_report.rename(index={
    'precision':'precision (with thresholds)',
    'recall':'recall (with thresholds)',
    'f1-score':'f1-score (with thresholds)'
},inplace=True)

without_threshold_report.rename(index={
    'precision':'precision (without thresholds)',
    'recall':'recall (without thresholds)',
    'f1-score':'f1-score (without thresholds)'
},inplace=True)

final_report = pd.concat([with_threshold_report,without_threshold_report])

In [ ]:
final_report.to_csv('result/scores/threshold_report.csv')

In [ ]:
final_report

### Second level Model

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pred = pd.DataFrame(columns=['prediction','pattern'])
c=0
for fold in proba_dist:
    print("Processing fold:",c)

    # train set
    X_meta_train = fold['meta_proba_train'].drop(columns=['pattern'])
    X_lr_train = fold['lr_proba_train'].drop(columns=['pattern'])
    X_nn_train = fold['nn_proba_train'].drop(columns=['pattern'])
    y_temp_train = fold['meta_proba_train']['pattern']

    X_meta_train = X_meta_train.add_prefix('meta_')
    X_lr_train = X_lr_train.add_prefix('lr_')
    X_nn_train = X_nn_train.add_prefix('nn_')

    # Validation set
    X_meta_val = fold['meta_proba_val'].drop(columns=['pattern'])
    X_lr_val = fold['lr_proba_val'].drop(columns=['pattern'])
    X_nn_val = fold['nn_proba_val'].drop(columns=['pattern'])
    y_temp_val = fold['meta_proba_val']['pattern']

    X_meta_val = X_meta_val.add_prefix('meta_')
    X_lr_val = X_lr_val.add_prefix('lr_')
    X_nn_val = X_nn_val.add_prefix('nn_')

    X_meta_val = pd.concat([X_meta_train,X_meta_val], axis=0)
    X_lr_val = pd.concat([X_lr_train,X_lr_val], axis=0)
    X_nn_val = pd.concat([X_nn_train,X_nn_val], axis=0)
    y_temp_val = pd.concat([y_temp_train,y_temp_val], axis=0)
    
    # Test set
    X_meta_test = fold['meta_proba_test'].drop(columns=['pattern'])
    X_lr_test = fold['lr_proba_test'].drop(columns=['pattern'])
    X_nn_test = fold['nn_proba_test'].drop(columns=['pattern'])
    y_temp_test = fold['meta_proba_test']['pattern']

    X_meta_test = X_meta_test.add_prefix('meta_')
    X_lr_test = X_lr_test.add_prefix('lr_')
    X_nn_test = X_nn_test.add_prefix('nn_')


    dataset_val = pd.concat([X_meta_val,X_nn_val,X_lr_val,y_temp_val], axis=1)
    dataset_test = pd.concat([X_meta_test,X_nn_test,X_lr_test,y_temp_test], axis=1)

    # display(dataset)

    lr_sec_model,le_sec_scaler,le_sec_encoder = lr_train(dataset_val)
    lr_sec_scaled = le_sec_scaler.transform(dataset_test.drop(columns=['pattern']))
    lr_sec_pred = pd.DataFrame(columns=['prediction','pattern'])
    lr_sec_pred['prediction'] = lr_sec_model.predict(lr_sec_scaled)
    lr_sec_pred['pattern'] = dataset_test['pattern'].values
    lr_pred = pd.concat([lr_pred,lr_sec_pred])
    c+=1

In [ ]:
report = classification_report(lr_pred['prediction'], lr_pred['pattern'],labels=lr_pred['pattern'].unique().tolist(), output_dict=True, zero_division=0)
pd.DataFrame(report).T

In [ ]:
def extract_class(col_name):
    # meta_Classical Models → Classical Models
    return col_name.split('_', 1)[1]

def normalize_weights(weights):
    s = sum(weights.values())
    return {k: v / s for k, v in weights.items()}

def weighted_aggregate(df, weights):
    """
    df      : DataFrame with columns like meta_*, nn_*, lr_*
    weights : dict like {"meta":0.4, "nn":0.35, "lr":0.25}
    """
    class_scores = {}

    for col in df.columns:
        prefix, cls = col.split('_', 1)
        w = weights.get(prefix, 0.0)

        if cls not in class_scores:
            class_scores[cls] = w * df[col]
        else:
            class_scores[cls] += w * df[col]

    return pd.DataFrame(class_scores)

MODEL_WEIGHTS = normalize_weights({
    "meta": 0.5,
    "nn":   0.3,
    "lr":   0.2
})

agg_pred = pd.DataFrame(columns=["prediction", "pattern"])

c = 0
for fold in proba_dist:
    print("Processing fold:", c)

    # -------- TEST --------
    X_meta_test = fold["meta_proba_test"].drop(columns=["pattern"]).add_prefix("meta_")
    X_nn_test   = fold["nn_proba_test"].drop(columns=["pattern"]).add_prefix("nn_")
    X_lr_test   = fold["lr_proba_test"].drop(columns=["pattern"]).add_prefix("lr_")
    y_test      = fold["meta_proba_test"]["pattern"]

    X_test = pd.concat([X_meta_test, X_nn_test, X_lr_test], axis=1)

    # -------- AGGREGATION + PREDICTION --------
    test_scores = weighted_aggregate(X_test, MODEL_WEIGHTS)
    y_pred      = test_scores.idxmax(axis=1)

    fold_pred = pd.DataFrame({
        "prediction": y_pred.values,
        "pattern": y_test.values
    })

    agg_pred = pd.concat([agg_pred, fold_pred], ignore_index=True)

    c += 1

IGNORE_CLASS = "Other"

labels = [c for c in agg_pred["pattern"].unique() if c != IGNORE_CLASS]

report = classification_report(
    agg_pred["pattern"],
    agg_pred["prediction"],
    labels=labels,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).T
macro = report["macro avg"]

print(
    f"Precision: {macro['precision']:.4f} | "
    f"Recall: {macro['recall']:.4f} | "
    f"F1: {macro['f1-score']:.4f}"
)
report_df

In [ ]:
X_meta_test